# Web Scraping — Top 1000 canales de Twitch

Extracción de datos desde [twitchtracker.com](https://twitchtracker.com), dividida en tres fases:

1. **Ranking general**: extracción del top 1000 canales con BeautifulSoup.
2. **Datos por canal**: contenido principal, idioma, fecha de creación y suscriptores.
3. **Histórico de streams**: últimos 40 streams por canal con Selenium.

Los datasets resultantes se guardan en la carpeta `data/` y son la entrada del notebook de análisis (`02_EDA.ipynb`).

---
## Fase 1 — Ranking general (Top 1000)

In [ ]:
from fake_useragent import UserAgent
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup as bs
import time
import random

ua = UserAgent()
headers = {'User-Agent': ua.random}

url = "https://twitchtracker.com/channels/ranking"

In [ ]:
response = requests.get(url, headers=headers)
soup = bs(response.content, 'html.parser')

dict_str = {}
dict_str['streamer'] = [x.get_text() for x in soup.find_all("a", target="_blank")[1::2]]
dict_str['href'] = [x.get("href") for x in soup.find_all("a", target="_blank")[1::2]]
dict_str['media visualizaciones'] = [x.get_text() for x in soup.find_all("td", class_="color-viewers")]
dict_str['horas stream'] = [x.get_text().split("hours")[0] for x in soup.find_all("td", class_="color-streamed")]
dict_str['maximo historico'] = [x.find("span").get_text() for x in soup.find_all("td",class_="color-viewersMax")]
dict_str['aumento followers'] = [x.find("span").get_text() for x in soup.find_all("td",class_="color-followers hidden-sm")[::2]]
dict_str['total followers'] = [x.find("span").get_text() for x in soup.find_all("td",class_="color-followers hidden-sm")[1::2]]

In [ ]:
for pagina in list(range(2,21)):

    url=str("https://twitchtracker.com/channels/ranking?page="+str(pagina))
    response = requests.get(url, headers=headers)
    soup = bs(response.content, 'html.parser')

    dict_str['streamer'] += [x.get_text() for x in soup.find_all("a", target="_blank")[1::2]]
    dict_str['href'] += [x.get("href") for x in soup.find_all("a", target="_blank")[1::2]]
    dict_str['media visualizaciones'] += [x.get_text() for x in soup.find_all("td", class_="color-viewers")]
    dict_str['horas stream'] += [x.get_text().split("hours")[0] for x in soup.find_all("td", class_="color-streamed")]
    dict_str['maximo historico'] += [x.find("span").get_text() for x in soup.find_all("td",class_="color-viewersMax")]
    dict_str['aumento followers'] += [x.find("span").get_text() for x in soup.find_all("td",class_="color-followers hidden-sm")[::2]]
    dict_str['total followers'] += [x.find("span").get_text() for x in soup.find_all("td",class_="color-followers hidden-sm")[1::2]]
    
    time.sleep(random.randint(3,7))

In [ ]:
streamers = pd.DataFrame(dict_str)
streamers

In [ ]:
streamers.to_csv("data/1000_Streamers.csv", index=False)

---
## Fase 2 — Datos por canal

Para cada canal extraemos el contenido principal, idioma, fecha de creación del canal y suscriptores. El proceso se divide en dos pasadas: la primera recoge la mayoría de los datos, y la segunda reintenta los canales donde `maximo subs` quedó vacío.

In [ ]:
from fake_useragent import UserAgent
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup as bs
import time
import random

ua = UserAgent()
headers = {'User-Agent': ua.random}

datos_streamers = pd.read_csv("data/1000_Streamers.csv")
nombre = list(datos_streamers.iloc[:,0].values)

In [ ]:
# Contenido principal de cada canal
for i, j in enumerate(nombre):

    try:
        url = "https://twitchtracker.com"+datos_streamers.iloc[i,1]+"/games"
        response = requests.get(url, headers=headers)
        soup = bs(response.content, 'html.parser')
        datos_streamers.loc[datos_streamers["streamer"]==j,"contenido principal"] = soup.find_all("td")[2].get_text()
        print("Scrapeo n",i,"Obteniendo datos de ",j)
        time.sleep(random.randint(3,6))
    except:
        continue

datos_streamers.to_csv("data/1000_Streamers_2.csv", index=False)

In [ ]:
# Idioma, fecha de creación y suscriptores
datos_streamers = pd.read_csv("data/1000_Streamers_2.csv")
nombre = list(datos_streamers.iloc[:,0].values)

for i, j in enumerate(nombre):

    try:
        url = "https://twitchtracker.com"+datos_streamers.iloc[i,1]
        response = requests.get(url, headers=headers)
        soup = bs(response.content, 'html.parser')
        print("Scrapeo n",i,"Obteniendo datos de ",j)

        datos_streamers.loc[datos_streamers["streamer"]==j,"idioma"] = soup.find_all("a", class_="label label-soft")[0].get_text()
        datos_streamers.loc[datos_streamers["streamer"]==j,"fecha creacion canal"] = soup.find_all("span", class_="label label-soft to-date")[0].get_text()[0:10]
        datos_streamers.loc[datos_streamers["streamer"]==j,"subs actuales"] = soup.find_all("div", class_="g-x-s-value to-number")[0].get_text()
        datos_streamers.loc[datos_streamers["streamer"]==j,"maximo subs"] = soup.find_all("div", class_="g-x-s-value to-number")[3].get_text()

        time.sleep(random.randint(2,5))
    except:
        print("petó")
        continue

datos_streamers.to_csv("data/1000_Streamers_3.csv", index=False)

In [ ]:
# Segunda pasada: reintentar canales con maximo subs vacío
datos_streamers = pd.read_csv("data/1000_Streamers_3.csv")
filtro = datos_streamers[(~datos_streamers["subs actuales"].isnull())&(datos_streamers["maximo subs"].isnull())]
nombre = list(filtro.iloc[:,0].values)

for i, j in enumerate(nombre):

    try:
        url = "https://twitchtracker.com"+filtro.iloc[i,1]
        response = requests.get(url, headers=headers)
        soup = bs(response.content, 'html.parser')
        print("Scrapeo n",i,"Obteniendo datos de ",j)

        datos_streamers.loc[datos_streamers["streamer"]==j,"maximo subs"] = soup.find_all("div", class_="g-x-s-value to-number")[-1].get_text()

        time.sleep(random.randint(3,6))
    except:
        print("petó")
        continue

datos_streamers.to_csv("data/1000_Streamers_3.csv", index=False)

---
## Fase 3 — Histórico de streams (Selenium)

Para cada canal extraemos los datos de sus últimos 40 streams: fecha, duración, media y máximo de espectadores, followers ganados y hasta 4 categorías de contenido por stream. Al tratarse de contenido dinámico es necesario usar Selenium antes de parsear con BeautifulSoup.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

datos_streamers = pd.read_csv("data/1000_Streamers_2.csv")
nombre = list(datos_streamers.iloc[:,0].values)

lista_fallos=('WarThunder_eSports','thek4sen','genesisgg','ow_esports__','MORGENSHTERN','Codonysus','JatyLito','Gustavin017','1GO_Casino','FRG_SR','Twitchmedia18','FishAtAce','湊あくあ_本物','rayasianboy','1_GO_Casino','bruxaofc01','FRG_stm','PimentoIa','FargoBossx','Twitchmedia13','ForexNowTV','Sabeky','twitchmedia54','twitchmedia6','xbuyeroficial','xFibii','Luxy_grlx','Luxury_qq','twitchmedia5','luchogal_10','Luxy_grl','FRG_FFS','KarmineCorp','Ilytog','LuxuryGirl_STRM','FRG_gj','BUXEXA_AO_VIVO','SlotHustla','Luxry_Fly','JakuLak','FRGxy')

for i in lista_fallos:
    nombre.remove(i)

dict={"streamer":[],
      "fecha":[],
     "duracion":[],
     "media viewers":[],
     "max viewers":[],
     "followers":[],
     "contenido 1":[],
     "contenido 2":[],
     "contenido 3":[],
     "contenido 4":[]}

lista_fallos=[]

In [ ]:
for num, streamer in enumerate(nombre):

    streamerb=[]  
    fecha=[]
    duracion=[]
    media_viewers=[]
    max_viewers=[]
    followers=[]
    contenido_1=[]
    contenido_2=[]
    contenido_3=[]
    contenido_4=[]

    try:
        service = Service(executable_path='./chromedriver.exe')

        chrome_options = Options()
        chrome_options.add_argument("--headless")
        driver = webdriver.Chrome(service=service, options=chrome_options)

        url = "https://twitchtracker.com"+datos_streamers.iloc[num,1]+"/streams"

        print("Obteniendo datos de:",streamer)
        driver.get(url)

        personaldata = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div[1]/div[3]/div[2]/button[1]/p')
        personaldata.click()

        soup = BeautifulSoup(driver.page_source, "html.parser")

        pares = []
        impares = []
        contador = 0

        for i in soup.find_all("tr", class_="even"):
            for h in i.find_all("span"):
                if h.get_text()[0]=="+":
                    continue
                else:
                    pares.append(h)

        for i in soup.find_all("tr", class_="odd"):
            for h in i.find_all("span"):
                if h.get_text()[0]=="+":
                    continue
                else:
                    impares.append(h)

        for h,j in enumerate(impares):
            
            contador+=1
            match contador:
                case 1:
                    fecha.append(j.get_text())
                    fecha.append(pares[h].get_text())
                case 2:
                    duracion.append(j.get_text())
                    duracion.append(pares[h].get_text())
                case 3:
                    media_viewers.append(j.get_text())
                    media_viewers.append(pares[h].get_text())
                case 4:
                    max_viewers.append(j.get_text())
                    max_viewers.append(pares[h].get_text())
                case 5:
                    followers.append(j.get_text())
                    followers.append(pares[h].get_text())
                case 6:
                    contador=0

        contador = 0
        for i in soup.find_all("td", class_="games hidden-xs hidden-sm"):

            match contador:
                case 1: 
                    contenido_2.append(pd.NA)
                    contenido_3.append(pd.NA)
                    contenido_4.append(pd.NA)
                case 2: 
                    contenido_3.append(pd.NA)
                    contenido_4.append(pd.NA)
                case 3: 
                    contenido_4.append(pd.NA) 

            contador = 0      

            for h, j in enumerate(i.find_all("img")):
                contador+=1
                
                match h:
                    case 0:
                        contenido_1.append(j.get("data-original-title"))
                    case 1:
                        contenido_2.append(j.get("data-original-title"))
                    case 2:
                        contenido_3.append(j.get("data-original-title"))
                    case 3:
                        contenido_4.append(j.get("data-original-title"))

        match contador:
            case 1: 
                contenido_2.append(pd.NA)
                contenido_3.append(pd.NA)
                contenido_4.append(pd.NA)
            case 2: 
                contenido_3.append(pd.NA)
                contenido_4.append(pd.NA)
            case 3: 
                contenido_4.append(pd.NA)

        if (len(pares)+len(impares))==120:
            while soup!=1:
                try:
                    pagina_2 = driver.find_element(By.XPATH,'/html/body/div[2]/div[4]/div[5]/div[2]/div/ul/li[2]/a')
                    pagina_2.click()
                    break
                except:
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    continue

            soup = BeautifulSoup(driver.page_source, "html.parser")
            pares = []
            impares = []
            contador = 0

            for i in soup.find_all("tr", class_="even"):
                for h in i.find_all("span"):
                    if h.get_text()[0]=="+":
                        continue
                    else:
                        pares.append(h)

            for i in soup.find_all("tr", class_="odd"):
                for h in i.find_all("span"):
                    if h.get_text()[0]=="+":
                        continue
                    else:
                        impares.append(h)

            for h,j in enumerate(impares):
                contador+=1
                match contador:
                    case 1:
                        fecha.append(j.get_text())
                        fecha.append(pares[h].get_text())
                    case 2:
                        duracion.append(j.get_text())
                        duracion.append(pares[h].get_text())
                    case 3:
                        media_viewers.append(j.get_text())
                        media_viewers.append(pares[h].get_text())
                    case 4:
                        max_viewers.append(j.get_text())
                        max_viewers.append(pares[h].get_text())
                    case 5:
                        followers.append(j.get_text())
                        followers.append(pares[h].get_text())
                    case 6:
                        contador=0

            contador = 0
            for i in soup.find_all("td", class_="games hidden-xs hidden-sm"):

                match contador:
                    case 1: 
                        contenido_2.append(pd.NA)
                        contenido_3.append(pd.NA)
                        contenido_4.append(pd.NA)
                    case 2: 
                        contenido_3.append(pd.NA)
                        contenido_4.append(pd.NA)
                    case 3: 
                        contenido_4.append(pd.NA) 

                contador = 0      

                for h, j in enumerate(i.find_all("img")):
                    contador+=1
                    
                    match h:
                        case 0:
                            contenido_1.append(j.get("data-original-title"))
                        case 1:
                            contenido_2.append(j.get("data-original-title"))
                        case 2:
                            contenido_3.append(j.get("data-original-title"))
                        case 3:
                            contenido_4.append(j.get("data-original-title"))

            match contador:
                case 1: 
                    contenido_2.append(pd.NA)
                    contenido_3.append(pd.NA)
                    contenido_4.append(pd.NA)
                case 2: 
                    contenido_2.append(pd.NA)
                    contenido_3.append(pd.NA)
                case 3: 
                    contenido_4.append(pd.NA)

            driver.close()

        resta = len(fecha)-len(streamerb)
        for i in list(range(resta)):
            streamerb.append(streamer)

        dict["streamer"] += streamerb
        dict["fecha"] += fecha
        dict["duracion"] += duracion
        dict["media viewers"] += media_viewers
        dict["max viewers"] += max_viewers
        dict["followers"] += followers
        dict["contenido 1"] += contenido_1
        dict["contenido 2"] += contenido_2
        dict["contenido 3"] += contenido_3
        dict["contenido 4"] += contenido_4

        time.sleep(random.randint(3,6))

    except:
        print("fallo")
        lista_fallos.append(streamer)

In [ ]:
pd.DataFrame(dict)

In [ ]:
pd.DataFrame(dict).to_csv("data/DATOS_STREAMS.csv", index=False)